## 4章 多言語の固有認識表現

##### 4.1 データセット
+ XTREME : Cross-lingual TRansfer Evaluation of Multilingal Encoders
+ PAN-Xを探す

In [1]:
from huggingface_hub import notebook_login
notebook_login()

In [2]:
from pprint import pprint
from datasets import get_dataset_config_names

In [3]:
xtreme_subsets = get_dataset_config_names("xtreme")
pprint(xtreme_subsets)
print(f"XTREME has {len(xtreme_subsets)} configurations")

['MLQA.ar.ar',
 'MLQA.ar.de',
 'MLQA.ar.en',
 'MLQA.ar.es',
 'MLQA.ar.hi',
 'MLQA.ar.vi',
 'MLQA.ar.zh',
 'MLQA.de.ar',
 'MLQA.de.de',
 'MLQA.de.en',
 'MLQA.de.es',
 'MLQA.de.hi',
 'MLQA.de.vi',
 'MLQA.de.zh',
 'MLQA.en.ar',
 'MLQA.en.de',
 'MLQA.en.en',
 'MLQA.en.es',
 'MLQA.en.hi',
 'MLQA.en.vi',
 'MLQA.en.zh',
 'MLQA.es.ar',
 'MLQA.es.de',
 'MLQA.es.en',
 'MLQA.es.es',
 'MLQA.es.hi',
 'MLQA.es.vi',
 'MLQA.es.zh',
 'MLQA.hi.ar',
 'MLQA.hi.de',
 'MLQA.hi.en',
 'MLQA.hi.es',
 'MLQA.hi.hi',
 'MLQA.hi.vi',
 'MLQA.hi.zh',
 'MLQA.vi.ar',
 'MLQA.vi.de',
 'MLQA.vi.en',
 'MLQA.vi.es',
 'MLQA.vi.hi',
 'MLQA.vi.vi',
 'MLQA.vi.zh',
 'MLQA.zh.ar',
 'MLQA.zh.de',
 'MLQA.zh.en',
 'MLQA.zh.es',
 'MLQA.zh.hi',
 'MLQA.zh.vi',
 'MLQA.zh.zh',
 'PAN-X.af',
 'PAN-X.ar',
 'PAN-X.bg',
 'PAN-X.bn',
 'PAN-X.de',
 'PAN-X.el',
 'PAN-X.en',
 'PAN-X.es',
 'PAN-X.et',
 'PAN-X.eu',
 'PAN-X.fa',
 'PAN-X.fi',
 'PAN-X.fr',
 'PAN-X.he',
 'PAN-X.hi',
 'PAN-X.hu',
 'PAN-X.id',
 'PAN-X.it',
 'PAN-X.ja',
 'PAN-X.jv',
 'PAN

In [4]:
panx_subsets = [s for s in xtreme_subsets if s.startswith("PAN")]
panx_subsets[:5]

['PAN-X.af', 'PAN-X.ar', 'PAN-X.bg', 'PAN-X.bn', 'PAN-X.de']

In [5]:
# ドイツ語のコーパスを取得
from datasets import load_dataset

tmp_ds = load_dataset("xtreme", name="PAN-X.de")
print(tmp_ds)
for it in tmp_ds:
    print(it)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 10000
    })
})
train
validation
test


In [6]:
# 各言語(ドイツ語, フランス語, イタリア語, 英語)のコーパスから
# 特定の割合でサンプリングしてデータセットを作る
from collections import defaultdict
from datasets import DatasetDict

langs = ["de", "fr", "it", "en"]
fracs = [0.629, 0.229, 0.084, 0.059]

# キーが存在しなければ, DatasetDictを返す
panx_ch = defaultdict(DatasetDict)
for lang, frac in zip(langs, fracs):
    # 単語コーパスをロード
    ds = load_dataset("xtreme", name=f"PAN-X.{lang}")
    # 各分割をシャッフルし、話者の割合に応じてダウンサンプリング
    for split in ds: # train, validation, test
        print(f"langs: {lang}, split: {split}")
        panx_ch[lang][split] = (
            ds[split]
            .shuffle(seed=0)
            .select(range(int(frac * ds[split].num_rows)))
        )


Using the latest cached version of the dataset since xtreme couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'PAN-X.de' at F:\HFCache\datasets\xtreme\PAN-X.de\0.0.0\ec5f1f46e9af79639a90684a7a70a956c4998f04 (last modified on Mon Sep 29 19:06:22 2025).


langs: de, split: train
langs: de, split: validation
langs: de, split: test


Using the latest cached version of the dataset since xtreme couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'PAN-X.fr' at F:\HFCache\datasets\xtreme\PAN-X.fr\0.0.0\ec5f1f46e9af79639a90684a7a70a956c4998f04 (last modified on Mon Sep 29 19:18:08 2025).


langs: fr, split: train
langs: fr, split: validation
langs: fr, split: test


Using the latest cached version of the dataset since xtreme couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'PAN-X.it' at F:\HFCache\datasets\xtreme\PAN-X.it\0.0.0\ec5f1f46e9af79639a90684a7a70a956c4998f04 (last modified on Mon Sep 29 19:18:27 2025).


langs: it, split: train
langs: it, split: validation
langs: it, split: test


Using the latest cached version of the dataset since xtreme couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'PAN-X.en' at F:\HFCache\datasets\xtreme\PAN-X.en\0.0.0\ec5f1f46e9af79639a90684a7a70a956c4998f04 (last modified on Mon Sep 29 19:18:43 2025).


langs: en, split: train
langs: en, split: validation
langs: en, split: test


In [7]:
print(panx_ch)

defaultdict(<class 'datasets.dataset_dict.DatasetDict'>, {'de': DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 12580
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 6290
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 6290
    })
}), 'fr': DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 4580
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 2290
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 2290
    })
}), 'it': DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 1680
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 840
    })
    test: Dataset({
        features: ['token

In [8]:
# 各データセットのnum_row属性にアクセスして学習データに含まれる言語ごとの事例を確認
import pandas as pd

pd.DataFrame({lang: [panx_ch[lang]["train"].num_rows] for lang in langs},
             index=["Number of training examples"])

,de,fr,it,en
Number of training examples,12580,4580,1680,1180


In [9]:
# ドイツ語コーパスに含まれる一つの事例を検証
element = panx_ch['de']['train'][0]
for key, value in element.items():
    print(f"{key}: {value}")

tokens: ['2.000', 'Einwohnern', 'an', 'der', 'Danziger', 'Bucht', 'in', 'der', 'polnischen', 'Woiwodschaft', 'Pommern', '.']
ner_tags: [0, 0, 0, 0, 5, 6, 0, 0, 5, 5, 6, 0]
langs: ['de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de']


In [10]:
for key, value in panx_ch["de"]["train"].features.items():
    print(f"{key}: {value}")

tokens: List(Value('string'))
ner_tags: List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']))
langs: List(Value('string'))


In [11]:
panx_ch["de"]["train"]

Dataset({
    features: ['tokens', 'ner_tags', 'langs'],
    num_rows: 12580
})

In [12]:
ner_tags = panx_ch["de"]["train"].features['ner_tags']
ner_tags

List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']))

In [13]:
ner_tags.feature

ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])

In [14]:
ner_tags.feature.__dict__

{'names': ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'],
 'id': None,
 'num_classes': 7,
 'names_file': None,
 '_int2str': ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'],
 '_str2int': {'O': 0,
  'B-PER': 1,
  'I-PER': 2,
  'B-ORG': 3,
  'I-ORG': 4,
  'B-LOC': 5,
  'I-LOC': 6}}

In [15]:
tags = panx_ch["de"]["train"].features["ner_tags"].feature
print("tag: ", tags)

def create_tag_names(batch):
    # tagsは int <-> str の変換ルール.
    return {"ner_tags_str": [tags.int2str(idx) for idx in batch["ner_tags"]]}

tag:  ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])


In [16]:
panx_ch["de"]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 12580
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 6290
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 6290
    })
})

In [17]:
panx_de = panx_ch["de"].map(create_tag_names)

Map:   0%|          | 0/12580 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

In [18]:
de_example = panx_de["train"][0] # trainデータの一つ目
pd.DataFrame([de_example["tokens"], de_example["ner_tags_str"]], ["Tokens", "Tags"])

,0,1,2,3,4,5,6,7,8,9,10,11
Tokens,2.000,Einwohnern,an,der,Danziger,Bucht,in,der,polnischen,Woiwodschaft,Pommern,.
Tags,O,O,O,O,B-LOC,I-LOC,O,O,B-LOC,B-LOC,I-LOC,O


In [29]:
# タグに異常な不均衡がないことを簡単に確認するために
# 各分割における各固有表現の頻度を計算する
from collections import Counter

split2freqs = defaultdict(Counter)
for split, dataset in panx_de.items():
    for row in dataset["ner_tags_str"]:
        for tag in row:
            if tag.startswith("B"):
                tag_type = tag.split("-")[1]
                split2freqs[split][tag_type] += 1

pd.DataFrame.from_dict(split2freqs, orient="index")

,LOC,ORG,PER
train,6186,5366,5810
validation,3172,2683,2893
test,3180,2573,3071


#### 4.2 多言語Transformer

SentencePiece VS WordPiece
+ XLM-R系モデルは生のテキストを直接トークン化するSentencePieceを使う.
+ 100言語の生テキストで学習させたSentencePieceを使う.
+ BERTはWordPieceを使っている?
+

In [19]:
# BERTとXML-Rをロード
from transformers import AutoTokenizer

bert_model_name = "bert-base-cased"
xlmr_model_name = "xlm-roberta-base"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
xlmr_tokenizer = AutoTokenizer.from_pretrained(xlmr_model_name)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

C:\Users\inoue\Documents\MyGithub\Book_Transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in F:\HFCache\hub\models--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

C:\Users\inoue\Documents\MyGithub\Book_Transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in F:\HFCache\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [20]:
# テキストをエンコードすることで、各モデルが事前学習時に使用した特殊トークンを取得できる
text = "Jack Sparrow loves New York!"
bert_tokens = bert_tokenizer(text).tokens()
xlmr_tokens = xlmr_tokenizer(text).tokens()
print(bert_tokens) # [CLS], ##, [SEP]
print(xlmr_tokens) # <s>, _***, </s>

['[CLS]', 'Jack', 'Spa', '##rrow', 'loves', 'New', 'York', '!', '[SEP]']
['<s>', '▁Jack', '▁Spar', 'row', '▁love', 's', '▁New', '▁York', '!', '</s>']


#### トークナイザーのパイプライン
1. 正規化 : 空白の除去, アクセント付き文字の除去, Unicode正規化(NFC,NFD,NFKC,NFKDを1つに統一), 小文字化
2. 事前トークン化 : BPE(Byte-Pair Encoding)やUnigramアルゴリズム, WordPieceなどでテキストをトークン列に分割. or 言語圏特有のライブラリの使用.
3. トークナイザーモデル : (学習済み)モデルでトークン列をサブワードに分割する. (サブワード分割モデルの適用)
4. 後処理 : 特殊トークンの追加など

In [21]:
# SentencePieceトークナイザー
"".join(xlmr_tokens) # _ : Lower One Quarter Block

'<s>▁Jack▁Sparrow▁loves▁New▁York!</s>'

In [22]:
"".join(xlmr_tokens).replace(u"\u2581", " ")

'<s> Jack Sparrow loves New York!</s>'

#### 固有表現認識用のTransformer